 # Masterclass in Agroclimatic Analytics: Harnessing NASA POWER with `aidweather`

 ### *A Unified Framework for Scalable, Cache-Backed Spatial-Temporal Analysis in Agronomy & Crop Modeling*

 **Author:** Cleverson Matiolli, PhD & Gemini
 **Ecosystem:** `aidweather` (Base Package) -> `aidviz` (Visualization) -> `aidfarm` (Ecosystem Analytics)

 ---

 ## Executive Overview & Biophysical Rationale

 Quantitative agronomy and precision agriculture require high-resolution, robust meteorological feeds to parameterize dynamic crop growth models (e.g., DSSAT, APSIM) and machine learning models for yield forecasting. However, raw environmental APIs present significant operational friction:
 1. **High Latency & Rate Limits:** Repetitive spatial-temporal queries can throttle IP addresses or violate API service level agreements (SLAs).
 2. **Inconsistent Geocoordinate Syntax:** Parsing Degrees-Minutes-Seconds (DMS), Degrees-Decimal Minutes (DDM), or raw Decimal Degrees (DD) from heterogeneous fieldwork logs introduces human and numeric errors.
 3. **Data Quality & Schema Drift:** Handling missing values, timezone misalignments, and irregular formats requires boilerplate data cleaning pipelines.

 The `aidweather` package solves these issues. This masterclass tutorial explores all aspects of `aidweather`:
 - **Centralized Configuration:** Bundled metadata catalogues and agronomic dictionaries.
 - **Geospatial Coordinate Normalization:** Value objects for converting, validating, and displaying geolocations.
 - **SQLite Caching Engine:** Intelligent split-and-merge database caching that reduces network overhead.
 - **Parallel Concurrency:** Thread-pool based spatial querying across coordinate grids.
 - **Data Lineage:** Preprocessing utilities to ensure robust Pandas time-series alignments.

 ---

 ## Biophysical & STEM-Level Weather Parameters

 The NASA POWER API utilizes the **MERRA-2 (Modern-Era Retrospective analysis for Research and Applications, Version 2)** data assimilation system and satellite observations to estimate key parameters:

 *   **Photosynthetically Active Radiation ($\text{PAR}$, code `ALLSKY_SFC_PAR_TOT`):**
     $$\text{PAR} = \int_{400\text{ nm}}^{700\text{ nm}} I(\lambda) d\lambda \quad \left[\text{MJ}\cdot\text{m}^{-2}\cdot\text{day}^{-1}\right]$$
     Drives the crop carbon accumulation rate. Crop growth engines utilize $\text{PAR}$ coupled with a Radiation Use Efficiency ($\text{RUE}$) parameter to calculate daily dry matter production: $\Delta\text{Biomass} = \text{PAR} \times \text{RUE}$.

 *   **Air Temperature at 2 meters ($T_{2\text{m}}$, code `T2M`, `T2M_MAX`, `T2M_MIN`):**
     Controls the phenological clock via Growing Degree Days ($\text{GDD}$):
     $$\text{GDD} = \max\left(\frac{T_{2\text{m, max}} + T_{2\text{m, min}}}{2} - T_{\text{base}}, 0\right)$$
     Large diurnal temperature ranges ($T_{\text{range}} = T_{2\text{m, max}} - T_{2\text{m, min}}$) alter the ratio of crop photosynthesis to respiration.

 *   **Dew Point Temperature at 2 meters ($T_{\text{dew}}$, code `T2MDEW`):**
     Serves as an operational proxy for Leaf Wetness Duration (LWD). Fungal spore germination (e.g., *Phakopsora pachyrhizi* or Soybean Rust) requires liquid water on the leaf surface, which occurs when the ambient temperature approaches the dew point:
     $$T_{2\text{m}} - T_{\text{dew}} \leq 2^{\circ}\text{C}$$

 *   **Corrected Total Precipitation ($P_{\text{corr}}$, code `PRECTOTCORR`):**
     A bias-adjusted estimate of rainfall and snowmelt, critical for calculating the field soil water balance:
     $$\Delta S = P_{\text{corr}} - \text{ET}_0 - D - R$$
     where $\text{ET}_0$ is the Reference Evapotranspiration, $D$ is deep drainage, and $R$ is surface runoff.

 Let's initialize our workspace and begin the demonstration!

In [1]:
import os
import sqlite3
import pandas as pd
import numpy as np
import time
from pathlib import Path

# Set up logging for interactive execution
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s"
)

# Import all core elements from aidweather
from aidweather import (
    PowerClient,
    GeoCoordinate,
    normalize_coord_input,
    cfg,
    get_config,
    ensure_date_column,
)

print("Packages loaded. Python environment and aidweather library are ready!")

Packages loaded. Python environment and aidweather library are ready!


 ## Part 1: Centralized Configuration System

 The `aidweather` package implements a centralized configuration architecture. It loads options from an internal, bundled `config.json` using `importlib.resources`. This mechanism provides a single source of truth across downstream applications (like `aidviz` and `aidfarm`), while falling back to safe defaults if the configuration file is modified or absent.

 Let's inspect the `cfg` singleton configuration values:

In [2]:
# Retrieve the configuration singleton instance
config = get_config()

# 1. Fetch default base URLs for NASA POWER API
print("=== API Endpoints ===")
print("Daily Point URL:", config.get_url("daily", "point"))
print("Hourly Point URL:", config.get_url("hourly", "point"))
print("Daily Regional URL:", config.get_url("daily", "regional"))

# 2. Access default agronomic parameter sets
print("\n=== Parameter Groups ===")
default_params = config.params("default")
print("Default Parameters ({}):".format(len(default_params)))
for code, name in default_params.items():
    print(f"  - {code:18s}: {name}")

=== API Endpoints ===
Daily Point URL: https://power.larc.nasa.gov/api/temporal/daily/point
Hourly Point URL: https://power.larc.nasa.gov/api/temporal/hourly/point
Daily Regional URL: https://power.larc.nasa.gov/api/temporal/daily/regional

=== Parameter Groups ===
Default Parameters (7):
  - T2M               : Temperature at 2 m (°C)
  - T2M_RANGE         : Temperature Range at 2 m (°C)
  - T2MDEW            : Dew Point Temperature at 2 m (°C)
  - RH2M              : Relative Humidity at 2 m (%)
  - PRECTOTCORR       : Corrected Total Precipitation (mm/day)
  - ALLSKY_SFC_PAR_TOT: All Sky Photosynthetically Active Radiation (MJ/m²/day)
  - WS10M             : Wind Speed at 10 m (m/s)


 Each parameter comes with a detailed biophysical description. Let's inspect the descriptions for agricultural features:

In [3]:
print("=== STEM Biophysical Descriptions ===")
descriptions = config.param_descriptions()
for param in ["T2M", "ALLSKY_SFC_PAR_TOT", "PRECTOTCORR", "T2MDEW"]:
    print(f"\n[{param}]:")
    print(descriptions.get(param, "No description found."))

=== STEM Biophysical Descriptions ===

[T2M]:
Average Air Temperature at 2 m (°C): Modeled daily average air temperature. Data is obtained from the NASA MERRA-2 assimilation model (1981-Present, ~0.5°x0.625° resolution). In agriculture, T2M is critical for calculating Growing Degree Days (GDD) and modeling crop developmental stages and growth rates. Used in machine learning models for yield prediction and thermal stress assessment.

[ALLSKY_SFC_PAR_TOT]:
All Sky Photosynthetically Active Radiation (MJ/m²/day): The total energy flux (400-700 nm) used by plants for photosynthesis. Data is derived from NASA satellite observations (1984-Present, ~1.0°x1.0° resolution). This directly quantifies the light available for plant growth. Used in Crop Growth Models (CGMs) like DSSAT and APSIM to simulate biomass accumulation and final yield.

[PRECTOTCORR]:
Corrected Total Precipitation (mm/day): Total liquid and frozen water precipitation, adjusted for known biases. Data is obtained from the NASA

 The configuration system also maps parameters to distinct hex colors to ensure consistent visualizations throughout the downstream ecosystem (e.g., `aidviz`).

 Let's view the hex map and visualize:

In [21]:
from IPython.display import HTML

# Get the color map from the configuration
color_map = config.color_map()

print("=== Parameter Color Map ===")
for param, hex_color in color_map.items():
    print(f"  - {param:18s}: {hex_color}")
    
print("\n=== Visualizing Parameter Color Map ===")
params = list(color_map.keys())
colors = list(color_map.values())

html = "<table style='width:100%; border-collapse:collapse;'>"
html += "<tr><th style='text-align:right; padding:10px;'>Parameter</th><th style='text-align:right;'>Color</th></tr>"
for param, color in zip(params, colors):
    html += f"<tr><td style='padding:10px; border:1px solid #ddd;'>{param}</td>"
    html += f"<td style='padding:10px; border:1px solid #ddd; background-color:{color}; width:640px;'></td></tr>"
html += "</table>"

display(HTML(html))

=== Parameter Color Map ===
  - T2M               : #D55E00
  - T2M_MAX           : #882255
  - T2M_MIN           : #56B4E9
  - T2M_RANGE         : #E69F00
  - T2MWET            : #CC79A7
  - T2MDEW            : #AA4499
  - TS                : #A0522D
  - RH2M              : #0072B2
  - PRECTOTCORR       : #009E73
  - ALLSKY_SFC_SW_DWN : #D4AC0D
  - ALLSKY_SFC_PAR_TOT: #117733
  - WS10M             : #808080
  - WS10M_MAX         : #000000
  - WD10M             : #999999
  - PS                : #332288

=== Visualizing Parameter Color Map ===


Parameter,Color
T2M,
T2M_MAX,
T2M_MIN,
T2M_RANGE,
T2MWET,
T2MDEW,
TS,
RH2M,
PRECTOTCORR,
ALLSKY_SFC_SW_DWN,


 ## Part 2: High-Precision Geospatial Coordinate Management

 Field notebooks and historical datasets often represent geographic locations in diverse formats:
 - **Decimal Degrees (DD):** `"-23.55"`
 - **Degrees, Decimal Minutes (DDM):** `"23° 33.0' S"`
 - **Degrees, Minutes, Seconds (DMS):** `"23° 33' 0\" S"`

 To guarantee type safety, value-object integrity, and error-free execution, `aidweather` provides the `GeoCoordinate` dataclass and the `normalize_coord_input` function. Let's test the robust parsing, validation, and conversion features:

In [22]:
# 1. Parsing diverse string representations
coords = [
    # Latitude in DDM, Longitude in DMS
    GeoCoordinate.from_strings("23° 33.0' S", "46° 37' 48\" W"),
    # Negative decimal numbers represented as strings
    GeoCoordinate.from_strings("-23.55°", "-46.63°"),
    # Raw numeric float values (South / West are negative)
    GeoCoordinate.from_decimal(-23.55, -46.63),
]

print("=== Parsed Coordinates ===")
for idx, c in enumerate(coords, 1):
    print(f"Coord {idx}: Lat: {c.lat:.5f}, Lon: {c.lon:.5f}")

# 2. Normalizing mixed inputs into a single GeoCoordinate object
mixed_input = normalize_coord_input((-23.55, "46°37'48.0\" W"))
print("\nNormalized Mixed Input (Tuple):", mixed_input)

=== Parsed Coordinates ===
Coord 1: Lat: -23.55000, Lon: -46.63000
Coord 2: Lat: -23.55000, Lon: -46.63000
Coord 3: Lat: -23.55000, Lon: -46.63000

Normalized Mixed Input (Tuple): GeoCoordinate(lat=-23.55, lon=-46.63)


 The `GeoCoordinate` object also lets you output coordinates back into standardized strings for publication-grade metadata files:

In [23]:
print("=== Exporting Formatted Strings ===")
print("Decimal Degrees (DD) :", mixed_input.to_dd_str(lat_precision=5))
print("Decimal Minutes (DDM):", mixed_input.to_ddm_str(minute_precision=3))
print("Minutes/Seconds (DMS):", mixed_input.to_dms_str(second_precision=2))

=== Exporting Formatted Strings ===
Decimal Degrees (DD) : ('23.55000° S', '46.63000° W')
Decimal Minutes (DDM): ("23°33.000' S", "46°37.800' W")
Minutes/Seconds (DMS): ('23°33\'0.00" S', '46°37\'48.00" W')


 The value object performs input range checks on instantiation to prevent submission of invalid coordinates to the NASA API, raising descriptive exceptions. Let's verify this validation:

In [24]:
try:
    # Latitude must reside between -90 and +90
    invalid_coord = GeoCoordinate.from_decimal(95.0, -46.63)
except ValueError as e:
    print("Caught expected validation error:", e)

Caught expected validation error: Latitude out of range [-90, 90]: 95.0


 ## Part 3: Production-Grade Weather Data Ingestion & Caching

 The `PowerClient` wraps the NASA POWER API with a robust SQLite database caching layer located at `~/.aidweather_cache/` by default (or configured inside `config.json`).

 When a weather query is submitted:
 1. The client checks the local cache database using a hashed representation of the request parameters and coordinates.
 2. It identifies whether the requested temporal bounds are already covered.
 3. If there is a missing sub-range, it fetches *only* the missing dates from the NASA API.
 4. It merges and deduplicates the cached and newly retrieved datasets, writes the merged data back to the database as a Gzip-compressed BLOB, and returns the requested range to the user.

 Let's demonstrate a full caching round-trip, measuring cold vs. hot request latencies.

 *Note: We will use a mock session or real requests depending on network status. Because `PowerClient` uses a local sqlite database, we will clear the cache to ensure we measure a true "cold" request.*

In [25]:
# Initialize PowerClient
client = PowerClient(temporal_api="daily")

# Check if cache is enabled and find the cache path
cache_info = client.cache_cfg
print("Cache Path:", cache_info.get("path"))
print("Cache Enabled:", cache_info.get("enabled"))

# Ensure cache is clear before running the benchmark
db_path = Path(cache_info.get("path", ".")) / "aidweather_cache.db"
if db_path.exists():
    print("Clearing historical cache to start benchmark...")
    try:
        db_path.unlink()
        print("Cache deleted.")
    except Exception as e:
        print(f"Could not delete cache file: {e}")

# Re-initialize client to create a clean database
client = PowerClient(temporal_api="daily")

2026-05-29 15:26:22,825 [INFO] NASA POWER API key detected and configured.


Info Note: NASA POWER API key detected and configured: ***iBoo

2026-05-29 15:26:22,833 [INFO] NASA POWER API key detected and configured.


Cache Path: .aidweather_cache
Cache Enabled: True
Clearing historical cache to start benchmark...
Cache deleted.


Info Note: NASA POWER API key detected and configured: ***iBoo

 Let's perform a **Cold Request** (fetching data for the first time). The client will fetch the weather metrics from the NASA servers.

In [26]:
# Define parameters for an agricultural parcel (e.g., Londrina, PR, Brazil - major agricultural hub)
lat, lon = -23.31, -51.16
start_date = "2023-01-01"
end_date = "2023-01-15"
params = ["T2M", "PRECTOTCORR", "ALLSKY_SFC_PAR_TOT"]

print(
    f"Executing Cold Request for coordinates ({lat}, {lon}) from {start_date} to {end_date}..."
)
t0 = time.perf_counter()

df_cold = client.get_point_data(
    lat=lat, lon=lon, start=start_date, end=end_date, params=params
)

t_cold = time.perf_counter() - t0
print(f"Cold Request completed in {t_cold:.3f} seconds.")
print("Shape of returned DataFrame:", df_cold.shape)
print(df_cold.head())

2026-05-29 15:26:26,014 [INFO] Fetching 1 missing date range(s).


Executing Cold Request for coordinates (-23.31, -51.16) from 2023-01-01 to 2023-01-15...


2026-05-29 15:26:26,858 [INFO] Updating cache for key a132fd992c63226f5976633b333e12ec12ab9c6aadafcf48bf881ed3e48282f1 with merged data.


Cold Request completed in 0.852 seconds.
Shape of returned DataFrame: (15, 3)
              T2M  PRECTOTCORR  ALLSKY_SFC_PAR_TOT
date                                              
2023-01-01  26.27         0.05               12.97
2023-01-02  25.37         3.67                4.12
2023-01-03  25.03         9.23               10.03
2023-01-04  22.69        12.33                5.72
2023-01-05  20.39         1.21                9.53


 Now, let's execute the exact same request again. Since the data is stored in the local cache, this will be a **Hot Request**, served entirely from the SQLite database.

In [27]:
print(f"Executing Hot Request for same coordinates ({lat}, {lon}) and dates...")
t0 = time.perf_counter()

df_hot = client.get_point_data(
    lat=lat, lon=lon, start=start_date, end=end_date, params=params
)

t_hot = time.perf_counter() - t0
print(f"Hot Request completed in {t_hot:.3f} seconds.")
print(f"Speedup factor: {t_cold / t_hot:.1f}x faster!")

2026-05-29 15:26:29,101 [INFO] Retrieved full date range from cache for key a132fd992c63226f5976633b333e12ec12ab9c6aadafcf48bf881ed3e48282f1.


Executing Hot Request for same coordinates (-23.31, -51.16) and dates...
Hot Request completed in 0.005 seconds.
Speedup factor: 176.1x faster!


 Let's execute the `summarize` function. This prints a formatted dashboard containing data insights, request statistics, network transfer performance, and connection states.

In [28]:
client.summarize(df_hot)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                      Weather Data Profile                                                                       │
│ ┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓                                                  │
│ ┃ Property            ┃ Value                                ┃                                                  │
│ ┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩                                                  │
│ │ Temporal Resolution │ Daily                                │                                                  │
│ │ Start Date          │ 2023-01-01 00:00:00                  │                                                  │
│ │ End Date            │ 2023-01-15 00:00:00                  │                                                  │
│ │ Data Points         │ 15                                   │                                                  │
│ │ Missing Values      │ 0 / 45                               │                                                  │
│ │ Parameters          │ T2M, PRECTOTCORR, ALLSKY_SFC_PAR_TOT │                                                  │
│ └─────────────────────┴──────────────────────────────────────┘                                                  │
╰───────────────────────────────────────────────── Data Insight ──────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│    Transfer & Cache Performance                                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓                                                                              │
│ ┃ Metric            ┃ Value      ┃                                                                              │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩                                                                              │
│ │ Network Duration  │ 0.84 s     │                                                                              │
│ │ Total Downloaded  │ 1.37 KiB   │                                                                              │
│ │ Avg Speed         │ 1.63 KiB/s │                                                                              │
│ │ Cache (Initial)   │ 324.00 B   │                                                                              │
│ │ Cache (Increment) │ 0.00 B     │                                                                              │
│ │ Cache (Total)     │ 324.00 B   │                                                                              │
│ └───────────────────┴────────────┘                                                                              │
╰────────────────────────────────────────────────── Performance ──────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│         Request Statistics                                                                                      │
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓                                                                              │
│ ┃ Metric                 ┃ Value ┃                                                                              │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩                                                                              │
│ │ Total Logical Requests │ 2     │                                                                              │
│ │ Cache Hits (Full)      │ 1     │                                                                              │
│ │ Network API Calls      │ 1     │                                                                              │
│ │ Cache Hit Rate         │ 50.0% │                                                                              │
│ └────────────────────────┴───────┘                                                                              │
╰────────────────────────────────────────────────── Efficiency ───────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│              NASA POWER Connection Info                                                                         │
│ ┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓                                                            │
│ ┃ Property       ┃ Value                           ┃                                                            │
│ ┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩                                                            │
│ │ API Key Status │ Provided (***iBoo)              │                                                            │
│ │ Auth Mode      │ Personal (Redacted)             │                                                            │
│ │ User Agent     │ aidweather/0.1.0                │                                                            │
│ │ Base URL       │ https://power.larc.nasa.gov/api │                                                            │
│ └────────────────┴─────────────────────────────────┘                                                            │
╰──────────────────────────────────────────────── API Connection ─────────────────────────────────────────────────╯

 Let's demonstrate the **Split-and-Merge Cache Logic**.
 We will query a date range that *partially* overlaps with the cached range: `"2023-01-10"` to `"2023-01-25"`.
 The client should identify that `2023-01-10` to `2023-01-15` is already cached, fetch *only* `2023-01-16` to `2023-01-25` from NASA, merge it, and cache the updated, continuous dataset.

In [29]:
extended_start = "2023-01-10"
extended_end = "2023-01-25"

print(f"Executing Overlapping Request from {extended_start} to {extended_end}...")
t0 = time.perf_counter()

df_extended = client.get_point_data(
    lat=lat, lon=lon, start=extended_start, end=extended_end, params=params
)

t_extended = time.perf_counter() - t0
print(f"Overlapping Request completed in {t_extended:.3f} seconds.")
print("Shape of extended DataFrame:", df_extended.shape)
print(
    "Index ranges returned: {} to {}".format(
        df_extended.index.min().date(), df_extended.index.max().date()
    )
)

2026-05-29 15:26:33,790 [INFO] Fetching 1 missing date range(s).


Executing Overlapping Request from 2023-01-10 to 2023-01-25...


2026-05-29 15:26:34,555 [INFO] Updating cache for key a132fd992c63226f5976633b333e12ec12ab9c6aadafcf48bf881ed3e48282f1 with merged data.


Overlapping Request completed in 0.774 seconds.
Shape of extended DataFrame: (16, 3)
Index ranges returned: 2023-01-10 to 2023-01-25


 ## Part 4: Advanced Concurrency & Spatial Transects

 Agricultural assessments often span multiple geographic points representing distinct treatment plots or an environmental gradient (e.g., altitude variation, distance from coastal humidity sources).

 The `PowerClient` offers two parallelized fetching routines:
 1. **`get_multi_point_data`:** Submits concurrent API queries across an arbitrary list of coordinate dictionaries or a Pandas DataFrame.
 2. **`get_expanded_point_data`:** Generates a linear sampling transect centered at a specific coordinate and fetches data along the spatial gradient.

 *NASA cautions against opening more than 5 concurrent HTTP connections from a single IP to prevent transient throttling. `PowerClient` handles thread execution limits safely and flags warnings if the concurrency pool exceeds standard guidelines.*

 Let's build a spatial transect running along the Latitude axis across a length of 20 kilometers:

In [30]:
center_lat, center_lon = -23.31, -51.16
distance_km = 20.0
num_points = 3  # Keep count low to respect NASA guidelines and speed up demo
transect_start = "2023-01-01"
transect_end = "2023-01-05"

print(
    f"Generating a {distance_km} km transect along the LATITUDE axis centered at ({center_lat}, {center_lon})..."
)

df_transect = client.get_expanded_point_data(
    lat=center_lat,
    lon=center_lon,
    start=transect_start,
    end=transect_end,
    params=["T2M", "PRECTOTCORR"],
    axis="lat",
    distance_km=distance_km,
    num_points=num_points,
    max_workers=3,
)

# Preview the returned multi-point spatial dataset
print("\nMulti-point transect dataframe shape:", df_transect.shape)
print("\nReturned DataFrame Head:")
print(df_transect.head(10))

# Notice that the returned DataFrame contains columns for 'lat' and 'lon', identifying each point!
print("\nUnique coordinates in dataset:")
print(df_transect[["lat", "lon"]].drop_duplicates().reset_index(drop=True))

2026-05-29 15:26:36,162 [INFO] Generated 3 points along the lat axis.
2026-05-29 15:26:36,165 [INFO] Fetching 1 missing date range(s).
2026-05-29 15:26:36,169 [INFO] Fetching 1 missing date range(s).
2026-05-29 15:26:36,169 [INFO] Fetching 1 missing date range(s).


Generating a 20.0 km transect along the LATITUDE axis centered at (-23.31, -51.16)...


2026-05-29 15:26:36,580 [INFO] Updating cache for key 1e3c5fdcb1af70d7e18db3f8c6fffe2d6d96fb7b0edfdff7bb67a7321ccfb580 with merged data.
2026-05-29 15:26:36,602 [INFO] Updating cache for key f3c3b22b14527a71b140cfb49f548081eb78f2c2e768aaee925a10f46fc14e49 with merged data.
2026-05-29 15:26:36,890 [INFO] Updating cache for key 4a0dc48c3f58cfc02374323861d053e00afe14007f3697dda33a92f6c70fbd2b with merged data.



Multi-point transect dataframe shape: (15, 6)

Returned DataFrame Head:
        date    T2M  PRECTOTCORR    lat    lon     name
0 2023-01-01  26.27         0.05 -23.31 -51.16  Point_2
1 2023-01-02  25.37         3.67 -23.31 -51.16  Point_2
2 2023-01-03  25.03         9.23 -23.31 -51.16  Point_2
3 2023-01-04  22.69        12.33 -23.31 -51.16  Point_2
4 2023-01-05  20.39         1.21 -23.31 -51.16  Point_2
5 2023-01-01  27.33         0.13 -23.22 -51.16  Point_3
6 2023-01-02  26.43        12.07 -23.22 -51.16  Point_3
7 2023-01-03  26.11        11.41 -23.22 -51.16  Point_3
8 2023-01-04  23.72        18.74 -23.22 -51.16  Point_3
9 2023-01-05  21.51         0.56 -23.22 -51.16  Point_3

Unique coordinates in dataset:
     lat    lon
0 -23.31 -51.16
1 -23.22 -51.16
2 -23.40 -51.16


 ## Part 5: Data Preprocessing & Alignment Lineage

 Raw weather datasets can contain inconsistencies (e.g., date formats under column labels such as `obs_date`, `time`, `RecordDate`, or date records set in DatetimeIndexes).
 To provide a reliable, clean transition from raw inputs to predictive ML pipelines or crop models, the `aidweather.utils` module includes the `ensure_date_column` function.

 Let's inspect how it robustly:
 1. Identifies date fields from a list of candidate names.
 2. Normalizes date times to timezone-naive midnight strings.
 3. Falls back to a DatetimeIndex when no date column is explicitly present.

In [31]:
# 1. Scenario A: DataFrame with a non-standard column name and string dates
raw_data_a = pd.DataFrame(
    {
        "obs_date": [
            "2023-05-15 08:30:00",
            "2023-05-16 12:45:00",
            "2023-05-17 19:15:00",
        ],
        "T2M": [18.5, 20.2, 17.8],
    }
)
print("Raw DataFrame A:")
print(raw_data_a)

# Standardize date column
clean_df_a = ensure_date_column(
    raw_data_a, name="date", candidates=["obs_date", "measurement_time"]
)
print("\nStandardized DataFrame A:")
print(clean_df_a)
print("Dtype of date column:", clean_df_a["date"].dtype)

Raw DataFrame A:
              obs_date   T2M
0  2023-05-15 08:30:00  18.5
1  2023-05-16 12:45:00  20.2
2  2023-05-17 19:15:00  17.8

Standardized DataFrame A:
    T2M       date
0  18.5 2023-05-15
1  20.2 2023-05-16
2  17.8 2023-05-17
Dtype of date column: datetime64[us]


 Let's demonstrate the fallback when the date is stored in the Index as a DatetimeIndex:

In [32]:
# 2. Scenario B: DataFrame with a DatetimeIndex and no standard columns
datetime_idx = pd.date_range("2023-06-01", periods=3, freq="D")
raw_data_b = pd.DataFrame({"RH2M": [65.0, 72.0, 58.0]}, index=datetime_idx)
print("Raw DataFrame B (DatetimeIndex):")
print(raw_data_b)

# Generate unified date column from index fallback
clean_df_b = ensure_date_column(raw_data_b, name="date", index_fallback=True)
print("\nStandardized DataFrame B:")
print(clean_df_b)

Raw DataFrame B (DatetimeIndex):
            RH2M
2023-06-01  65.0
2023-06-02  72.0
2023-06-03  58.0

Standardized DataFrame B:
            RH2M       date
2023-06-01  65.0 2023-06-01
2023-06-02  72.0 2023-06-02
2023-06-03  58.0 2023-06-03


 ---

 ## Conclusion & Operational Guidelines

 We have completed our masterclass in weather data ingestion and preprocessing using `aidweather`!

 ### Key Architecture Takeaways:
 *   **Performance Optimization:** Always leverage the local caching layer. For production servers, set the `path` option in `config.json` or through environment variables to persistent disk storage to prevent cold startup API lags.
 *   **Scientific Credibility:** Coordinate handling must preserve spatial precision. Use `GeoCoordinate` to parse input types cleanly and avoid geographic offsets.
 *   **Parallel Query Control:** Keep parallel worker threads under the recommended safety maximum ($N=5$) when submitting requests to NASA to prevent rate limits.
 *   **Ecosystem Integration:** Standardized date columns using `ensure_date_column` represent a critical interface boundary. Clean dataframes can be directly fed into the downstream `aidviz` visualization routines or downstream machine learning tasks in `aidfarm`.